# Exercise 3: Temperature sampling

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Cross-Lingual Transfer · version 0.1.0*

> Test temperature sampling on an imbalanced multilingual training set and report effective exposure and per-language development scores.

**Setup.** `pip install sklearn-crfsuite scikit-learn sacrebleu torch transformers sentencepiece fasttext-wheel "stanza==1.11.0" requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Data come from commit-pinned GitHub repositories and are hash-checked: KazNERD and KyrgyzNER (Exercise 1), SIB-200 (Exercises 2 and 3), the X-WMT Kazakh–Kyrgyz test set (Exercise 4) and the UD 2.17 Turkish–German SAGT and Turkish IMST test files (Exercise 5). Sentence encoders and NLLB-200 are read from local copies (`MODELS_DIR`, `HF_HUB_DIR`, `NLLB_PATH`) or the Hub at the revisions pinned in the code; GlotLID from `GLOTLID_PATH`. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's helpers from `../code/remaining/transfer.py`. Embeddings, translations and model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

The pool has 500 Turkish, 100 Kazakh, 40 Kyrgyz, 20 Tatar and 10 Bashkir SIB-200 training sentences, all different items. Training sets of 2,000 draws are sampled with the chapter's $p_i \propto n_i^{1/T}$ for four temperatures and five seeds; the classifier (NLLB encoder vectors, logistic regression) is scored on every language's development split.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, sys
from pathlib import Path
import numpy as np, requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from transfer import SCHEMA, bio_spans, conversion_map, restore_span

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/xwmt/test/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
    UD.format("UD_Turkish_German-SAGT", "qtd_sagt-ud-test.conllu"): "0bbb06de66a490bfbb93d103a03d0d89e3835baf0af755a57b5fa5f885ae01bc",
    UD.format("UD_Turkish-IMST", "tr_imst-ud-test.conllu"): "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Script view (TurkicNLP transliterator); Turkish and English stay unchanged."""
    if lang in ("tur", "eng"):
        return text
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = Script.LATIN if lang in ("uzb", "aze") else Script.CYRILLIC
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def sib(lang):
    """All 1,004 SIB-200 rows of a language as (index_id, split, topic, text), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), split, r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return sorted(rows)

MODELS = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
          "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a"}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
NLLB_PATH = os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()))
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl", "uzn_Latn": "uzn_Latn",
             "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl", "eng_Latn": "eng_Latn", "fra_Latn": "fra_Latn", "azj_Latn": "azj_Latn"}
def model_source(name):
    rev = MODELS[name]
    for p in (Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser() / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              Path(os.environ.get("MODELS_DIR", CACHE / "models")) / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev
_ENC = {}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors, as in Meaning and Retrieval (same cache)."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            _ENC[model] = (AutoTokenizer.from_pretrained(NLLB_PATH), AutoModelForSeq2SeqLM.from_pretrained(NLLB_PATH).get_encoder().eval(), 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def macro_f1(gold, pred, labels):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=labels, average="macro", zero_division=0)

print("helpers loaded from code/remaining/transfer.py; schema map:", SCHEMA)

In [ ]:
from sklearn.linear_model import LogisticRegression
LANGS = {"tur_Latn": 500, "kaz_Cyrl": 100, "kir_Cyrl": 40, "tat_Cyrl": 20, "bak_Cyrl": 10}      # an imbalanced pool
ROWS = {l: sib(l) for l in LANGS}
LABELS = sorted({r[2] for r in ROWS["tur_Latn"]})
split = [r[1] for r in ROWS["tur_Latn"]]; y = np.array([LABELS.index(r[2]) for r in ROWS["tur_Latn"]])
train_ids = [i for i, s in enumerate(split) if s == "train"]; dev_ids = [i for i, s in enumerate(split) if s == "dev"]
random.Random(0).shuffle(train_ids)
pool, start = {}, 0
for l, n in LANGS.items():                    # disjoint sentences per language, so no content is repeated across languages
    pool[l] = train_ids[start:start + n]; start += n
V = {l: embed("nllb", [r[3] for r in ROWS[l]], l) for l in LANGS}
def probs(T):
    w = {l: (len(ids) ** (1 / T) if T != "inf" else 1.0) for l, ids in pool.items()}
    z = sum(w.values()); return {l: v / z for l, v in w.items()}
print("training pool:", {l: len(v) for l, v in pool.items()}, "sentences (disjoint SIB-200 training items); budget 2,000 draws")
print(f"{'T':<6}" + "".join(f"{l[:3]:>8}" for l in LANGS) + "   | dev macro-F1 per language (mean of 5 seeds)" )
for T in (1, 2, 5, "inf"):
    p = probs(T); f1 = collections.defaultdict(list); exposure = collections.Counter()
    for seed in range(5):
        rng = np.random.default_rng(seed)
        langs = rng.choice(list(LANGS), size=2000, p=[p[l] for l in LANGS])
        X, Y = [], []
        for l in langs:
            i = pool[l][rng.integers(len(pool[l]))]; X.append(V[l][i]); Y.append(y[i]); exposure[l] += 1
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(np.array(X), np.array(Y))
        for l in LANGS:
            f1[l].append(macro_f1(y[dev_ids], clf.predict(V[l][dev_ids]), list(range(len(LABELS)))))
    share = "".join(f"{exposure[l] / 10000:8.1%}" for l in LANGS)
    print(f"{str(T):<6}{share}   | " + "  ".join(f"{l[:3]} {statistics.mean(f1[l]):.3f}" for l in LANGS))
uniq = {T: {l: round(len(pool[l]) * (1 - (1 - probs(T)[l] / len(pool[l])) ** 2000)) for l in LANGS} for T in (1, 5)}
print("expected distinct sentences seen per language at T=1:", uniq[1], " at T=5:", uniq[5])

## Worked solution

Output of `experiments/ch16_exercises/run_ch16_exercises.py`:

```
training pool: {'tur_Latn': 500, 'kaz_Cyrl': 100, 'kir_Cyrl': 40, 'tat_Cyrl': 20, 'bak_Cyrl': 10} sentences (disjoint SIB-200 training items); budget 2,000 draws
T          tur     kaz     kir     tat     bak   | dev macro-F1 per language (mean of 5 seeds)
1        74.2%   15.2%    5.9%    3.4%    1.3%   | tur 0.831  kaz 0.832  kir 0.825  tat 0.868  bak 0.818
2        48.1%   21.6%   13.8%    9.3%    7.2%   | tur 0.828  kaz 0.820  kir 0.793  tat 0.842  bak 0.815
5        30.4%   21.6%   18.1%   15.8%   14.1%   | tur 0.796  kaz 0.795  kir 0.770  tat 0.813  bak 0.740
inf      20.5%   19.7%   19.8%   19.6%   20.4%   | tur 0.774  kaz 0.782  kir 0.735  tat 0.786  bak 0.709
expected distinct sentences seen per language at T=1: {'tur_Latn': 475, 'kaz_Cyrl': 95, 'kir_Cyrl': 38, 'tat_Cyrl': 19, 'bak_Cyrl': 9}  at T=5: {'tur_Latn': 351, 'kaz_Cyrl': 99, 'kir_Cyrl': 40, 'tat_Cyrl': 20, 'bak_Cyrl': 10}
```

**Higher temperature gives smaller pools more draws.** At T=1, Turkish receives 74.2% and Bashkir 1.3% of draws, averaged across five seeds. Uniform language sampling gives each roughly 20%. Development macro-F1 is lower at higher tested temperatures for every language in this setup; Bashkir falls from 0.818 to 0.709.

The languages contribute different training sentences. Resampling therefore changes content, label frequencies and repetition as well as language exposure. The expected number of distinct Turkish items falls from 475 at T=1 to 351 at T=5. This is one possible contributor, not a demonstrated cause of the score change.

The encoder is frozen and only the logistic-regression classifier is trained. These results do not establish the best sampling policy for encoder fine-tuning or other datasets. Report variation across seeds as well as their means, then select a policy on development data.

## Your turn — extension

Repeat with the MiniLM encoder (weaker for Kyrgyz, Tatar and Bashkir) and with overlapping pools (the same sentences in every language). Does the best temperature change?